In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt

def visualize_class_images(root_dir, class_name, max_images=16):
    
    class_path = os.path.join(root_dir, class_name)
    
    image_files = [
        f for f in os.listdir(class_path)
        if f.endswith(".tif")
    ]
    
    image_files = image_files[:max_images]
    
    n = len(image_files)
    cols = 4
    rows = (n + cols - 1) // cols
    
    plt.figure(figsize=(12, 3*rows))
    
    for i, img_file in enumerate(image_files):
        img = tiff.imread(os.path.join(class_path, img_file))
        
        img = (img - img.min()) / (img.max() - img.min())
        
        plt.subplot(rows, cols, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(img_file)
        plt.axis("off")
    
    plt.suptitle(f"Virus: {class_name}", fontsize=16)
    plt.tight_layout()
    plt.show()



In [ ]:
train_root = "/kaggle/input/virus-images/context_virus_RAW/train"

classes = sorted([
    d for d in os.listdir(train_root)
    if os.path.isdir(os.path.join(train_root, d))
])

for cls in classes:
    visualize_class_images(train_root, cls, max_images=16)


In [ ]:
import os
import tifffile as tiff
import numpy as np
import matplotlib.pyplot as plt

def visualize_one_per_class(root_dir):
    
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])
    
    n_classes = len(classes)
    
    cols = 5
    rows = (n_classes + cols - 1) // cols
    
    plt.figure(figsize=(15, 3*rows))
    
    for i, cls in enumerate(classes):
        class_path = os.path.join(root_dir, cls)
        
        # Pick first tif image
        image_files = [
            f for f in os.listdir(class_path)
            if f.endswith(".tif")
        ]
        
        if len(image_files) == 0:
            continue
        
        img_path = os.path.join(class_path, image_files[0])
        img = tiff.imread(img_path)
        
        img = (img - img.min()) / (img.max() - img.min())
        
        plt.subplot(rows, cols, i+1)
        plt.imshow(img, cmap='gray')
        plt.title(cls, fontsize=9)
        plt.axis("off")
    
    plt.suptitle("Virus Overview", fontsize=18)
    plt.tight_layout()
    plt.show()


In [ ]:
train_root = "/kaggle/input/virus-images/context_virus_RAW/train"
visualize_one_per_class(train_root)


In [ ]:
plt.savefig("virus_overview.png", dpi=600, bbox_inches="tight")


In [ ]:
# Install if needed
!pip install timm torchmetrics seaborn -q


In [ ]:
import os
import torch
import timm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from tqdm import tqdm
from PIL import Image
import tifffile as tiff

from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.metrics import *
from sklearn.preprocessing import label_binarize
from torchmetrics.detection.mean_ap import MeanAveragePrecision

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


In [ ]:
def create_folders():
    folders = [
        "results",
        "results/classification",
        "results/detection",
        "results/hybrid",
        "results/roc",
        "results/confusion",
        "results/reports",
        "results/training_curves",
        "results/statistics"
    ]
    for f in folders:
        os.makedirs(f, exist_ok=True)

create_folders()


In [ ]:
import os

def create_result_folders():
    folders = [
        "results",
        "results/confusion",
        "results/roc",
        "results/reports",
        "results/training_curves"
    ]
    
    for folder in folders:
        os.makedirs(folder, exist_ok=True)

create_result_folders()


In [ ]:
def dataset_statistics(root_dir):
    classes = sorted([d for d in os.listdir(root_dir)
                      if os.path.isdir(os.path.join(root_dir, d))])

    stats = []

    for cls in classes:
        cls_path = os.path.join(root_dir, cls)
        num_images = len([f for f in os.listdir(cls_path) if f.endswith(".tif")])
        stats.append((cls, num_images))

    df = pd.DataFrame(stats, columns=["Class", "Images"])
    return df

train_root = "/kaggle/input/virus-images/context_virus_RAW/train"
df_stats = dataset_statistics(train_root)
print(df_stats)


In [ ]:
plt.figure(figsize=(10,6))
plt.bar(df_stats["Class"], df_stats["Images"])
plt.xticks(rotation=90)
plt.title("Class Distribution")
plt.tight_layout()
plt.show()


In [ ]:
!pip install torch torchvision timm tifffile scikit-learn tqdm -q

**Dataset Loader**

In [ ]:
import os
import tifffile as tiff
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np

class VirusClassificationDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for img in os.listdir(cls_path):
                if img.endswith(".tif"):
                    self.samples.append(
                        (os.path.join(cls_path, img), self.class_to_idx[cls])
                    )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        img = tiff.imread(img_path)
        
        # normalize
        img = (img - img.min()) / (img.max() - img.min())
        img = (img * 255).astype(np.uint8)
        
        img = Image.fromarray(img).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
            
        return img, label


**Transformers**

In [ ]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


**DataLoaders**

In [ ]:
from torch.utils.data import DataLoader

train_dataset = VirusClassificationDataset("/kaggle/input/virus-images/context_virus_RAW/train", transform)
val_dataset = VirusClassificationDataset("/kaggle/input/virus-images/context_virus_RAW/validation", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


**Model**

In [ ]:
'''import timm
import torch.nn as nn

model = timm.create_model(
    "swin_small_patch4_window7_224",
    pretrained=True,
    num_classes=22
)

model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
'''

**Handle Class Imbalance**

In [ ]:
from collections import Counter
import numpy as np

labels = [label for _, label in train_dataset.samples]
class_counts = Counter(labels)

weights = 1. / torch.tensor([class_counts[i] for i in range(22)], dtype=torch.float)
class_weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)


**Training Loop**

In [ ]:
'''import torch.optim as optim
from tqdm import tqdm

optimizer = optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(20):
    model.train()
    total_loss = 0
    
    for imgs, labels in tqdm(train_loader):
        imgs, labels = imgs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")
'''

****Virus Particle Detection****

***Custom Detection Dataset (Faster R-CNN)***

In [ ]:
import os
import torch
import tifffile as tiff
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.transforms import functional as F


In [ ]:
class VirusDetectionDataset(Dataset):
    def __init__(self, root_dir, box_size=64):
        self.root_dir = root_dir
        self.box_size = box_size
        
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        
        self.class_to_idx = {cls: i+1 for i, cls in enumerate(self.classes)}
        # +1 because 0 is background
        
        self.samples = []
        
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for file in os.listdir(cls_path):
                if file.endswith(".tif"):
                    self.samples.append((cls, os.path.join(cls_path, file)))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        cls, img_path = self.samples[idx]
        label = self.class_to_idx[cls]
        
        # ---- Load Image ----
        img = tiff.imread(img_path)
        img = (img - img.min()) / (img.max() - img.min())
        img = (img * 255).astype(np.uint8)
        img = Image.fromarray(img).convert("RGB")
        img = F.to_tensor(img)
        
        h, w = img.shape[1:]
        
        # ---- Build Annotation Path ----
        filename = os.path.splitext(os.path.basename(img_path))[0]
        txt_filename = filename + "_particlepositions.txt"
        
        txt_path = os.path.join(
            os.path.dirname(img_path),
            "particle_positions",
            txt_filename
        )
        
        boxes = []

        if os.path.exists(txt_path):
            with open(txt_path) as f:
                for line in f:
                    line = line.strip()
                    
                    # Skip empty lines
                    if line == "":
                        continue
                    
                    # Skip header lines
                    if not any(char.isdigit() for char in line):
                        continue
                    
                    # Only process lines with ";"
                    if ";" not in line:
                        continue
                    
                    parts = line.split(";")
                    
                    if len(parts) != 2:
                        continue
                    
                    try:
                        x = int(float(parts[0]))
                        y = int(float(parts[1]))
                    except:
                        continue
                    
                    x1 = max(0, x - self.box_size // 2)
                    y1 = max(0, y - self.box_size // 2)
                    x2 = min(w, x + self.box_size // 2)
                    y2 = min(h, y + self.box_size // 2)
                    
                    boxes.append([x1, y1, x2, y2])

        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
        
        labels = torch.ones((len(boxes),), dtype=torch.int64) * label
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        
        return img, target


**Collate Function**

In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))


**Create Data Loader**

In [ ]:
train_dataset = VirusDetectionDataset(
    "/kaggle/input/virus-images/context_virus_RAW/train",
    box_size=64
)

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)


In [ ]:
sample_txt = "/kaggle/input/virus-images/context_virus_RAW/train/TBE/particle_positions/5918_particlepositions.txt"

with open(sample_txt) as f:
    for i in range(10):
        print(f.readline())


In [ ]:
images, targets = next(iter(train_loader))

print(images[0].shape)
print(targets[0])


In [ ]:
images, targets = next(iter(train_loader))

print(images[0].shape)
print(targets[0]["boxes"].shape)
print(targets[0]["labels"].shape)


**Load Faster R-CNN**

In [ ]:
'''device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = 23

model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")

in_features = model.roi_heads.box_predictor.cls_score.in_features

model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
    in_features,
    num_classes
)

model.to(device)
'''

**Optimizer**

In [ ]:
#optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

**Detection Training Loop**

In [ ]:
'''from tqdm import tqdm

num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0
    
    for images, targets in tqdm(train_loader):
        
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        epoch_loss += losses.item()
    
    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader)}")
'''

In [ ]:
'''model.eval()

images, targets = next(iter(train_loader))
images = [img.to(device) for img in images]

with torch.no_grad():
    predictions = model(images)

print(predictions[0])
'''

You successfully built:

✔ Multi-class virus detection
✔ Faster R-CNN training
✔ Bounding box conversion from center annotations
✔ GPU-ready pipeline

> ****Count Images Per Class****

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

root = "/kaggle/input/virus-images/context_virus_RAW/train"

class_counts = {}

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if os.path.isdir(cls_path):
        count = len([f for f in os.listdir(cls_path) if f.endswith(".tif")])
        class_counts[cls] = count

df = pd.DataFrame(class_counts.items(), columns=["Class", "Image_Count"])
df = df.sort_values("Image_Count")

plt.figure()
plt.barh(df["Class"], df["Image_Count"])
plt.title("Class Distribution (Images)")
plt.xlabel("Number of Images")
plt.ylabel("Virus Class")
plt.show()


**Visualize Sample Image**

In [ ]:
import tifffile as tiff
from PIL import Image
import numpy as np

sample_path = os.path.join(root, list(class_counts.keys())[0])
sample_img = [f for f in os.listdir(sample_path) if f.endswith(".tif")][0]

img = tiff.imread(os.path.join(sample_path, sample_img))

plt.figure()
plt.imshow(img, cmap='gray')
plt.title("Sample TEM Image")
plt.axis("off")
plt.show()


**Visualize Particle Centers**

In [ ]:
import matplotlib.pyplot as plt

txt_path = os.path.join(
    sample_path,
    "particle_positions",
    sample_img.replace(".tif", "_particlepositions.txt")
)

centers = []

with open(txt_path) as f:
    for line in f:
        if ";" in line:
            x, y = line.strip().split(";")
            centers.append((int(float(x)), int(float(y))))

plt.figure()
plt.imshow(img, cmap='gray')

for (x, y) in centers:
    plt.scatter(x, y, s=10)

plt.title("Particle Centers")
plt.axis("off")
plt.show()


**Convert Centers → Bounding Boxes (Visualization)**

In [ ]:
box_size = 64

plt.figure()
plt.imshow(img, cmap='gray')

for (x, y) in centers:
    x1 = x - box_size//2
    y1 = y - box_size//2
    
    rect = plt.Rectangle((x1, y1),
                         box_size,
                         box_size,
                         fill=False)
    plt.gca().add_patch(rect)

plt.title("Bounding Boxes")
plt.axis("off")
plt.show()


**Particle Count Distribution**

In [ ]:
particle_counts = []

for cls in os.listdir(root):
    cls_path = os.path.join(root, cls)
    if not os.path.isdir(cls_path):
        continue
    
    for file in os.listdir(cls_path):
        if file.endswith(".tif"):
            txt_path = os.path.join(
                cls_path,
                "particle_positions",
                file.replace(".tif", "_particlepositions.txt")
            )
            count = 0
            if os.path.exists(txt_path):
                with open(txt_path) as f:
                    for line in f:
                        if ";" in line:
                            count += 1
            particle_counts.append(count)

plt.figure()
plt.hist(particle_counts)
plt.title("Distribution of Particle Counts Per Image")
plt.xlabel("Number of Particles")
plt.ylabel("Frequency")
plt.show()


***IMAGE-LEVEL CLASSIFICATION***

**Classification Dataset**

In [ ]:
import os
import tifffile as tiff
import numpy as np
from PIL import Image
import torch
from torch.utils.data import Dataset
from torchvision import transforms

class VirusClassificationDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for file in os.listdir(cls_path):
                if file.endswith(".tif"):
                    self.samples.append(
                        (os.path.join(cls_path, file),
                         self.class_to_idx[cls])
                    )
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        
        img = tiff.imread(img_path)
        img = (img - img.min()) / (img.max() - img.min())
        img = (img * 255).astype(np.uint8)
        img = Image.fromarray(img).convert("RGB")
        
        if self.transform:
            img = self.transform(img)
        
        return img, label


**Multi-scale Transform**

In [ ]:
import random

class MultiScaleResize:
    def __init__(self, sizes=[224, 256, 288]):
        self.sizes = sizes
        
    def __call__(self, img):
        size = random.choice(self.sizes)
        return transforms.Resize((size, size))(img)


**Data Augmentation**

In [ ]:
train_transform = transforms.Compose([
    MultiScaleResize([224, 256, 288]),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5]*3, std=[0.5]*3)
])


**Create Train / Val / Test Loaders**

In [ ]:
from torch.utils.data import DataLoader

train_dataset = VirusClassificationDataset(
    "/kaggle/input/virus-images/context_virus_RAW/train",
    train_transform
)

val_dataset = VirusClassificationDataset(
    "/kaggle/input/virus-images/context_virus_RAW/validation",
    val_transform
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)


**Metrics Function**

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc as sk_auc
)

from sklearn.preprocessing import label_binarize


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import *

def evaluate_model(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    #auc = roc_auc_score(labels, probs, multi_class="ovr")
    
    return acc, precision, recall, f1


In [ ]:
class_names = train_loader.dataset.classes

In [ ]:
import timm
import torch.nn as nn

def get_resnet():
    model = timm.create_model("resnet50", pretrained=True, num_classes=22)
    return model


In [ ]:
def get_efficientnet():
    model = timm.create_model("tf_efficientnetv2_s",
                              pretrained=True,
                              num_classes=22)
    return model


In [ ]:
def get_convnext():
    model = timm.create_model("convnext_small",
                              pretrained=True,
                              num_classes=22)
    return model


In [ ]:
def get_vit():
    model = timm.create_model("vit_base_patch16_224",
                              pretrained=True,
                              num_classes=22)
    return model


In [ ]:
def get_deit():
    model = timm.create_model("deit_base_patch16_224",
                              pretrained=True,
                              num_classes=22)
    return model


In [ ]:
def get_swin():
    model = timm.create_model("swin_small_patch4_window7_224",
                              pretrained=True,
                              num_classes=22)
    return model


In [ ]:
model = get_swin().to(device)
x = torch.randn(2,3,224,224).to(device)
print(model(x).shape)


**Training Function**

In [ ]:
import torch.optim as optim
from tqdm import tqdm

def train_model(model, train_loader, val_loader, device, epochs=10):
    
    model.to(device)
    
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    criterion = nn.CrossEntropyLoss()
    
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for imgs, labels in tqdm(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        acc, precision, recall, f1 = evaluate_model(model, val_loader, device)
        
        print(f"Epoch {epoch+1}")
        print("Loss:", total_loss/len(train_loader))
        print("Val Acc:", acc, "F1:", f1)
        
        history.append([acc, precision, recall, f1])
    
    return model, history


**Handle Class Imbalance Properly**

In [ ]:
from collections import Counter
import torch

labels = [label for _, label in train_dataset.samples]
class_counts = Counter(labels)

num_classes = 22

weights = []
for i in range(num_classes):
    weights.append(1.0 / class_counts[i])

weights = torch.tensor(weights, dtype=torch.float)
weights = weights / weights.sum() * num_classes


**Improved Training Function (Research Version)**

This version includes:

Weighted loss

Cosine scheduler

Best model saving

Full metric tracking

In [ ]:
import copy
import torch.optim as optim
from tqdm import tqdm
import torch.nn as nn

def train_model(model, model_name, train_loader, val_loader, device, epochs=15):
    
    model.to(device)
    
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    
    best_f1 = 0
    best_model = None
    
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for imgs, labels in tqdm(train_loader):
            imgs, labels = imgs.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
        
        scheduler.step()
        
        acc, precision, recall, f1 = evaluate_model(model, val_loader, device)
        
        print(f"\nModel: {model_name}")
        print(f"Epoch {epoch+1}/{epochs}")
        print("Train Loss:", total_loss/len(train_loader))
        print("Val Acc:", acc)
        print("Val Precision:", precision)
        print("Val Recall:", recall)
        print("Val F1:", f1)
        
        history.append({
            "epoch": epoch+1,
            "loss": total_loss/len(train_loader),
            "acc": acc,
            "precision": precision,
            "recall": recall,
            "f1": f1
        })
        
        if f1 > best_f1:
            best_f1 = f1
            best_model = copy.deepcopy(model.state_dict())
    
    model.load_state_dict(best_model)
    torch.save(model.state_dict(), f"{model_name}_best.pth")
    
    return model, history


**Train All 6 Models Sequentially**

In [ ]:
models_dict = {
    "ResNet50": get_resnet(),
    "EfficientNetV2": get_efficientnet(),
    "ConvNeXt": get_convnext(),
    "ViT": get_vit(),
    "Swin": get_swin(),
    "DeiT": get_deit()
}


In [ ]:
class_names = train_loader.dataset.classes
print(class_names)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

all_results = {}

for name, model in models_dict.items():
    print("\n==============================")
    print("Training:", name)
    print("==============================\n")
    
    trained_model, history = train_model(
        model,
        name,
        train_loader,
        val_loader,
        device,
        epochs=15
    )
    
    all_results[name] = history


**Final Comparison Table**

In [ ]:
import pandas as pd

final_results = []

for name, history in all_results.items():
    best_epoch = max(history, key=lambda x: x["f1"])
    final_results.append({
        "Model": name,
        "Accuracy": best_epoch["acc"],
        "Precision": best_epoch["precision"],
        "Recall": best_epoch["recall"],
        "F1 Score": best_epoch["f1"]
    })

results_df = pd.DataFrame(final_results)
results_df = results_df.sort_values("F1 Score", ascending=False)
print(results_df)


results_df.to_csv("results/classification_comparison.csv", index=False)


**Visualization of Model Comparison**

In [ ]:
models_dict = {
    "ResNet50": get_resnet(),
    "EfficientNetV2": get_efficientnet(),
    "ConvNeXt": get_convnext(),
    "ViT": get_vit(),
    "Swin": get_swin(),
    "DeiT": get_deit()
}

for name, model in models_dict.items():
    model.load_state_dict(torch.load(f"{name}_best.pth"))
    model.to(device)
    model.eval()


In [ ]:
class_names = train_dataset.classes
print(class_names)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np
import os

os.makedirs("results/confusion", exist_ok=True)

def plot_confusion_matrix(model, loader, device, model_name, class_names):
    
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(14,12))
    sns.heatmap(
        cm,
        annot=False,
        fmt="d",
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{model_name} Confusion Matrix")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    plt.savefig(f"results/confusion/{model_name}_confusion_matrix.png", dpi=300)
    plt.show()
    
    return cm


In [ ]:
for name, model in models_dict.items():
    print(f"\nGenerating confusion matrix for {name}")
    plot_confusion_matrix(
        model,
        val_loader,
        device,
        name,
        class_names
    )


You now have:

✔ Quantitative comparison
✔ Visual comparison
✔ Saved best models
✔ Reproducible experiment


**Load All Best Models**

In [ ]:
'''torch.save(best_model, f"results/{model_name}_best.pth")
'''

In [ ]:
import os
import torch

loaded_models = {}

for name, model in models_dict.items():

    weight_path = f"results/{name}_best.pth"

    if os.path.exists(weight_path):

        model.load_state_dict(
            torch.load(weight_path, map_location=device)
        )

        model = model.to(device)
        model.eval()

        loaded_models[name] = model

        print(f"✅ {name} loaded successfully.")

    else:
        print(f"⚠️ Warning: {weight_path} not found.")


**Equal Soft Voting (All 6 Models)**

In [ ]:
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np

os.makedirs("results/confusion", exist_ok=True)

def evaluate_full_ensemble(models_dict, loader, device, class_names):
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            ensemble_probs = None
            
            for model in models_dict.values():
                model.eval()
                outputs = model(imgs)
                probs = F.softmax(outputs, dim=1)
                
                if ensemble_probs is None:
                    ensemble_probs = probs
                else:
                    ensemble_probs += probs
            
            ensemble_probs /= len(models_dict)
            
            preds = torch.argmax(ensemble_probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # ---- Metrics ----
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    # ---- Confusion Matrix ----
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(14,12))
    sns.heatmap(
        cm,
        annot=False,
        cmap="Blues",
        xticklabels=class_names,
        yticklabels=class_names
    )
    
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Full Ensemble (6 Models) Confusion Matrix")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    
    plt.savefig("results/confusion/Full_Ensemble_confusion_matrix.png", dpi=300)
    plt.show()
    
    return acc, precision, recall, f1


**Evaluate**

In [ ]:
acc, precision, recall, f1 = evaluate_full_ensemble(
    models_dict,
    val_loader,
    device,
    class_names
)

print("Full Ensemble (6 Models)")
print("Accuracy:", acc)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)


**Create Weights from Results**

In [ ]:
model_weights = {
    "ResNet50": 0.604839,
    "EfficientNetV2": 0.766129,
    "ConvNeXt": 0.806452,
    "ViT": 0.564516,
    "Swin": 0.810484,
    "DeiT": 0.875000
}

# Normalize weights
total = sum(model_weights.values())
for k in model_weights:
    model_weights[k] /= total


**Weighted Ensemble Evaluation**

In [ ]:
def evaluate_weighted_ensemble(models_dict, weights_dict, loader, device):
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            labels = labels.to(device)
            
            ensemble_probs = None
            
            for name, model in models_dict.items():
                outputs = model(imgs)
                probs = F.softmax(outputs, dim=1)
                
                weight = weights_dict[name]
                
                if ensemble_probs is None:
                    ensemble_probs = weight * probs
                else:
                    ensemble_probs += weight * probs
            
            preds = torch.argmax(ensemble_probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    return acc, precision, recall, f1


**Run Weighted Ensemble**

In [ ]:
acc_w, precision_w, recall_w, f1_w = evaluate_weighted_ensemble(
    models_dict,
    model_weights,
    val_loader,
    device
)

print("Weighted Ensemble (6 Models)")
print("Accuracy:", acc_w)
print("Presicion:",precision_w)
print("Recall:",recall_w)
print("F1:", f1_w)


**Add to Comparison Table**

In [ ]:
results_df.loc[len(results_df)] = [
    "Full Ensemble (Equal)",
    acc,
    precision,
    recall,
    f1
]

results_df.loc[len(results_df)] = [
    "Full Ensemble (Weighted)",
    acc_w,
    precision_w,
    recall_w,
    f1_w
]

results_df = results_df.sort_values("F1 Score", ascending=False)
print(results_df)


**Function to Get Predictions**

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def get_predictions(model, loader, device):
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            preds = torch.argmax(outputs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_labels), np.array(all_preds)


**Ensemble Predictions Function**

In [ ]:
def get_weighted_ensemble_predictions(models_dict, weights_dict, loader, device):
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            ensemble_probs = None
            
            for name, model in models_dict.items():
                outputs = model(imgs)
                probs = torch.softmax(outputs, dim=1)
                
                weight = weights_dict[name]
                
                if ensemble_probs is None:
                    ensemble_probs = weight * probs
                else:
                    ensemble_probs += weight * probs
            
            preds = torch.argmax(ensemble_probs, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return np.array(all_labels), np.array(all_preds)


**Ensemble Confusion Matrix**

In [ ]:
true_labels_w, weighted_preds = get_weighted_ensemble_predictions(
    models_dict,
    model_weights,
    val_loader,
    device
)

cm_weighted = confusion_matrix(true_labels_w, weighted_preds)


**Plot Confusion Matrix (Raw Counts)**

In [ ]:
import os
os.makedirs("results/confusion", exist_ok=True)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm_weighted,
    annot=False,
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Weighted Ensemble (6 Models) Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()

plt.savefig("results/confusion/Weighted_Ensemble_confusion_matrix.png", dpi=300)
plt.show()


**Equal Ensemble Probabilities**

In [ ]:
def get_equal_ensemble_probs(models_dict, loader, device):
    
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            ensemble_probs = None
            
            for model in models_dict.values():
                outputs = model(imgs)
                probs = torch.softmax(outputs, dim=1)
                
                if ensemble_probs is None:
                    ensemble_probs = probs
                else:
                    ensemble_probs += probs
            
            ensemble_probs /= len(models_dict)
            
            all_probs.append(ensemble_probs.cpu())
            all_labels.extend(labels.numpy())
    
    return torch.cat(all_probs).numpy(), np.array(all_labels)


**Weighted Ensemble Probabilities**

In [ ]:
def get_weighted_ensemble_probs(models_dict, weights_dict, loader, device):
    
    all_probs = []
    all_labels = []
    
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            
            ensemble_probs = None
            
            for name, model in models_dict.items():
                outputs = model(imgs)
                probs = torch.softmax(outputs, dim=1)
                
                weight = weights_dict[name]
                
                if ensemble_probs is None:
                    ensemble_probs = weight * probs
                else:
                    ensemble_probs += weight * probs
            
            all_probs.append(ensemble_probs.cpu())
            all_labels.extend(labels.numpy())
    
    return torch.cat(all_probs).numpy(), np.array(all_labels)


**Compute ROC Curves (22 Classes)**

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import matplotlib.pyplot as plt
import os

os.makedirs("results/roc", exist_ok=True)

equal_probs, y_true = get_equal_ensemble_probs(models_dict, val_loader, device)
weighted_probs, _ = get_weighted_ensemble_probs(models_dict, model_weights, val_loader, device)

# Binarize labels
y_bin = label_binarize(y_true, classes=range(22))

plt.figure(figsize=(10,8))

# Micro-average ROC
fpr_eq, tpr_eq, _ = roc_curve(y_bin.ravel(), equal_probs.ravel())
roc_auc_eq = auc(fpr_eq, tpr_eq)

fpr_w, tpr_w, _ = roc_curve(y_bin.ravel(), weighted_probs.ravel())
roc_auc_w = auc(fpr_w, tpr_w)

plt.plot(fpr_eq, tpr_eq, label=f"Equal Ensemble (AUC = {roc_auc_eq:.3f})")
plt.plot(fpr_w, tpr_w, label=f"Weighted Ensemble (AUC = {roc_auc_w:.3f})")

plt.plot([0,1], [0,1], 'k--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Equal vs Weighted Ensemble ROC (Micro-average)")
plt.legend()
plt.grid(alpha=0.3)

plt.savefig("results/roc/Equal_vs_Weighted_ROC.png", dpi=300)
plt.show()


***Per-Class ROC (All 22)***

In [ ]:
plt.figure(figsize=(12,10))

for i in range(22):
    fpr, tpr, _ = roc_curve(y_bin[:, i], weighted_probs[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f"{class_names[i]} ({roc_auc:.2f})")

plt.plot([0,1],[0,1],'k--')
plt.legend(fontsize=6)
plt.title("Weighted Ensemble Per-Class ROC")
plt.savefig("results/roc/Weighted_PerClass_ROC.png", dpi=300)
plt.show()


**VISUAL COMPARISON BAR CHART**

In [ ]:
import pandas as pd

comparison_df = pd.DataFrame([
    {"Type": "Equal", "Accuracy": acc, "F1": f1},
    {"Type": "Weighted", "Accuracy": acc_w, "F1": f1_w}
])

comparison_df.set_index("Type").plot(kind="bar", figsize=(6,5))
plt.title("Equal vs Weighted Ensemble Comparison")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/statistics/Equal_vs_Weighted_Bar.png", dpi=300)
plt.show()


**OBJECT DETECTION BENCHMARKING**

**Proper Detection Dataset (With Validation)**

In [ ]:
import os
import torch
import tifffile as tiff
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from torchvision.transforms import functional as F
import random

class VirusDetectionDataset(Dataset):
    def __init__(self, root_dir, box_size=64):
        self.root_dir = root_dir
        self.box_size = box_size
        
        self.classes = sorted([
            d for d in os.listdir(root_dir)
            if os.path.isdir(os.path.join(root_dir, d))
        ])
        
        # background = 0
        self.class_to_idx = {cls: i+1 for i, cls in enumerate(self.classes)}
        
        self.samples = []
        
        for cls in self.classes:
            cls_path = os.path.join(root_dir, cls)
            for file in os.listdir(cls_path):
                if file.endswith(".tif"):
                    self.samples.append((cls, os.path.join(cls_path, file)))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        cls, img_path = self.samples[idx]
        label = self.class_to_idx[cls]
        
        # -----------------------------
        # Load Image
        # -----------------------------
        img = tiff.imread(img_path)
        img = (img - img.min()) / (img.max() - img.min())
        img = (img * 255).astype(np.uint8)
        
        img = Image.fromarray(img).convert("RGB")
        img = F.to_tensor(img)  # Now Tensor
        
        h, w = img.shape[1:]
        
        # -----------------------------
        # Load Annotation
        # -----------------------------
        filename = os.path.splitext(os.path.basename(img_path))[0]
        txt_filename = filename + "_particlepositions.txt"
        
        txt_path = os.path.join(
            os.path.dirname(img_path),
            "particle_positions",
            txt_filename
        )
        
        boxes = []
        
        if os.path.exists(txt_path):
            with open(txt_path) as f:
                for line in f:
                    line = line.strip()
                    
                    if ";" not in line:
                        continue
                    
                    parts = line.split(";")
                    
                    if len(parts) != 2:
                        continue
                    
                    try:
                        x = int(float(parts[0]))
                        y = int(float(parts[1]))
                    except:
                        continue
                    
                    x1 = max(0, x - self.box_size // 2)
                    y1 = max(0, y - self.box_size // 2)
                    x2 = min(w, x + self.box_size // 2)
                    y2 = min(h, y + self.box_size // 2)
                    
                    boxes.append([x1, y1, x2, y2])
        
        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        
        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
        
        labels = torch.ones((len(boxes),), dtype=torch.int64) * label
        
        # -----------------------------
        # Augmentation (Safe)
        # -----------------------------
        if random.random() > 0.5:
            img = torch.flip(img, dims=[2])  # horizontal
            
            if len(boxes) > 0:
                boxes[:, [0,2]] = w - boxes[:, [2,0]]
        
        if random.random() > 0.5:
            img = torch.flip(img, dims=[1])  # vertical
            
            if len(boxes) > 0:
                boxes[:, [1,3]] = h - boxes[:, [3,1]]
        
        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        
        return img, target


In [ ]:
def collate_fn(batch):
    return tuple(zip(*batch))


In [ ]:
train_det_dataset = VirusDetectionDataset(
    "/kaggle/input/virus-images/context_virus_RAW/train",
    box_size=64
)

val_det_dataset = VirusDetectionDataset(
    "/kaggle/input/virus-images/context_virus_RAW/validation",
    box_size=64
)

train_det_loader = DataLoader(
    train_det_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)

val_det_loader = DataLoader(
    val_det_dataset,
    batch_size=4,
    shuffle=False,
    collate_fn=collate_fn
)


**Add Detection Augmentation**

In [ ]:
train_det_loader = DataLoader(
    train_det_dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn
)


In [ ]:
images, targets = next(iter(train_det_loader))


In [ ]:
images, targets = next(iter(train_det_loader))
print(images[0].shape)
print(targets[0]["boxes"].shape)


In [ ]:
!pip install torchmetrics -q


**mAP Metric**

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

def evaluate_map(model, loader, device):
    
    model.eval()
    metric = MeanAveragePrecision(iou_type="bbox")
    
    with torch.no_grad():
        for images, targets in loader:
            
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            outputs = model(images)
            
            metric.update(outputs, targets)
    
    results = metric.compute()
    
    return results


**Faster R-CNN**

In [ ]:
import torchvision

num_classes = 23

faster_rcnn = torchvision.models.detection.fasterrcnn_resnet50_fpn(
    weights="DEFAULT"
)

in_features = faster_rcnn.roi_heads.box_predictor.cls_score.in_features

faster_rcnn.roi_heads.box_predictor = \
    torchvision.models.detection.faster_rcnn.FastRCNNPredictor(
        in_features,
        num_classes
    )

faster_rcnn.to(device)


**RetinaNet**

In [ ]:
import torchvision

num_classes = 23  # 22 classes + background

retinanet = torchvision.models.detection.retinanet_resnet50_fpn(
    weights="DEFAULT"
)

# Replace classification head properly
in_features = retinanet.head.classification_head.cls_logits.in_channels
num_anchors = retinanet.head.classification_head.num_anchors

retinanet.head.classification_head = \
    torchvision.models.detection.retinanet.RetinaNetClassificationHead(
        in_channels=in_features,
        num_anchors=num_anchors,
        num_classes=num_classes
    )

retinanet.to(device)


**DETR (Transformer Detection)**

In [ ]:
!pip install --upgrade torchvision -q


**Generic Training Function (Detection)**

In [ ]:
def train_detection_model(model, name, train_loader, val_loader, device, epochs=10):
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        
        for images, targets in train_loader:
            
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            optimizer.zero_grad()
            losses.backward()
            optimizer.step()
            
            total_loss += losses.item()
        
        print(f"{name} Epoch {epoch+1} Loss:", total_loss/len(train_loader))
        
        results = evaluate_map(model, val_loader, device)
        print(f"{name} mAP:", results["map"].item())
        print(f"{name} mAP@50:", results["map_50"].item())
    
    return model


**Train Sequentially**

In [ ]:
faster_rcnn = train_detection_model(
    faster_rcnn,
    "Faster R-CNN",
    train_det_loader,
    val_det_loader,
    device,
    epochs=10
)

retinanet = train_detection_model(
    retinanet,
    "RetinaNet",
    train_det_loader,
    val_det_loader,
    device,
    epochs=10
)



**Compare Results Table**

In [ ]:
results_frcnn = evaluate_map(
    faster_rcnn,
    val_det_loader,
    device
)

results_retina = evaluate_map(
    retinanet,
    val_det_loader,
    device
)


In [ ]:
import pandas as pd

det_results = pd.DataFrame([
    {
        "Model": "Faster R-CNN",
        "mAP": results_frcnn["map"].item(),
        "mAP@50": results_frcnn["map_50"].item()
    },
    {
        "Model": "RetinaNet",
        "mAP": results_retina["map"].item(),
        "mAP@50": results_retina["map_50"].item()
    }
])

print(det_results)


**Visualize Comparison**

In [ ]:
import matplotlib.pyplot as plt

plt.figure()
plt.bar(det_results["Model"], det_results["mAP@50"])
plt.title("Detection Model Comparison (mAP@50)")
plt.ylabel("mAP@50")
plt.show()


**Visualize Detection Outputs**

In [ ]:
def visualize_predictions(model, loader, device):
    model.eval()
    
    images, targets = next(iter(loader))
    images_device = [img.to(device) for img in images]
    
    with torch.no_grad():
        outputs = model(images_device)
    
    img = images[0].permute(1,2,0).numpy()
    
    plt.figure(figsize=(6,6))
    plt.imshow(img)
    
    boxes = outputs[0]["boxes"].cpu()
    scores = outputs[0]["scores"].cpu()
    
    for box, score in zip(boxes, scores):
        if score > 0.5:
            x1, y1, x2, y2 = box
            rect = plt.Rectangle(
                (x1, y1),
                x2-x1,
                y2-y1,
                fill=False
            )
            plt.gca().add_patch(rect)
    
    plt.title("Predictions")
    plt.axis("off")
    plt.show()


In [ ]:
visualize_predictions(faster_rcnn, val_det_loader, device)
visualize_predictions(retinanet, val_det_loader, device)


**Yolo**

In [ ]:
!pip install ultralytics -q


In [ ]:
import os
import tifffile as tiff
import numpy as np
from PIL import Image

def convert_to_yolo_format(root_dir, output_dir, box_size=64):
    
    os.makedirs(os.path.join(output_dir, "images"), exist_ok=True)
    os.makedirs(os.path.join(output_dir, "labels"), exist_ok=True)
    
    classes = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])
    
    class_to_idx = {cls: i for i, cls in enumerate(classes)}
    
    for cls in classes:
        cls_path = os.path.join(root_dir, cls)
        
        for file in os.listdir(cls_path):
            if not file.endswith(".tif"):
                continue
            
            img_path = os.path.join(cls_path, file)
            img = tiff.imread(img_path)
            
            h, w = img.shape
            
            # Save image as jpg (YOLO prefers jpg/png)
            img_out_path = os.path.join(output_dir, "images", file.replace(".tif", ".jpg"))
            Image.fromarray(img).convert("RGB").save(img_out_path)
            
            # Prepare label file
            label_filename = file.replace(".tif", ".txt")
            label_path = os.path.join(output_dir, "labels", label_filename)
            
            txt_filename = file.replace(".tif", "_particlepositions.txt")
            txt_path = os.path.join(cls_path, "particle_positions", txt_filename)
            
            with open(label_path, "w") as f_out:
                
                if os.path.exists(txt_path):
                    with open(txt_path) as f:
                        for line in f:
                            if ";" not in line:
                                continue
                            
                            x, y = line.strip().split(";")
                            x = float(x)
                            y = float(y)
                            
                            x_center = x / w
                            y_center = y / h
                            width = box_size / w
                            height = box_size / h
                            
                            class_id = class_to_idx[cls]
                            
                            f_out.write(
                                f"{class_id} {x_center} {y_center} {width} {height}\n"
                            )


**Convert Train and Val**

In [ ]:
convert_to_yolo_format(
    "/kaggle/input/virus-images/context_virus_RAW/train",
    "yolo_dataset/train"
)

convert_to_yolo_format(
    "/kaggle/input/virus-images/context_virus_RAW/validation",
    "yolo_dataset/val"
)


**Create data.yaml**

In [ ]:
yaml_content = """
path: yolo_dataset
train: train/images
val: val/images

nc: 22

names:
"""

for cls in train_dataset.classes:
    yaml_content += f"  - {cls}\n"

with open("virus.yaml", "w") as f:
    f.write(yaml_content)


**Train YOLO26**

In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("yolo26m.pt") 

yolo_model.train(
    data="virus.yaml",
    epochs=50,
    imgsz=800,
    batch=8
)


***Hybrid Detection + Classification System***

**Freeze Faster R-CNN (Inference Mode)**

In [ ]:
faster_rcnn.eval()


**Crop Detected Particles**

In [ ]:
def get_particle_crops(model, image_tensor, device, score_thresh=0.5):
    
    model.eval()
    
    with torch.no_grad():
        outputs = model([image_tensor.to(device)])
    
    boxes = outputs[0]["boxes"]
    scores = outputs[0]["scores"]
    
    crops = []
    
    for box, score in zip(boxes, scores):
        if score > score_thresh:
            x1, y1, x2, y2 = box.int()
            crop = image_tensor[:, y1:y2, x1:x2]
            crops.append(crop)
    
    return crops


**Use Full Ensemble Classifier**

In [ ]:
import torch.nn.functional as F

def classify_crops(crops, models_dict, device):
    
    final_probs = []
    
    for crop in crops:
        
        if crop.shape[1] < 10 or crop.shape[2] < 10:
            continue
        
        crop = F.interpolate(
            crop.unsqueeze(0),
            size=(224,224)
        )
        
        ensemble_probs = None
        
        for model in models_dict.values():
            model.eval()
            output = model(crop.to(device))
            probs = torch.softmax(output, dim=1)
            
            if ensemble_probs is None:
                ensemble_probs = probs
            else:
                ensemble_probs += probs
        
        ensemble_probs /= len(models_dict)
        final_probs.append(ensemble_probs)
    
    return final_probs


**Aggregate to Final Image Prediction**

In [ ]:
def aggregate_predictions(particle_probs):
    
    if len(particle_probs) == 0:
        return None
    
    total = sum(particle_probs)
    avg = total / len(particle_probs)
    
    final_class = torch.argmax(avg, dim=1)
    
    return final_class.item()


**Full Hybrid Inference**

In [ ]:
def hybrid_inference(image_tensor, detector, classifier_models, device):
    
    crops = get_particle_crops(detector, image_tensor, device)
    
    particle_probs = classify_crops(crops, classifier_models, device)
    
    final_class = aggregate_predictions(particle_probs)
    
    return final_class


**FINAL EVALUATION — Hybrid vs Pure Classification**

**Hybrid Evaluation Loop**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def evaluate_hybrid(detector, classifier_models, val_loader, device):
    
    detector.eval()
    
    all_preds = []
    all_labels = []
    
    for imgs, labels in val_loader:
        
        for img, label in zip(imgs, labels):
            
            pred = hybrid_inference(img, detector, classifier_models, device)
            
            # If no detection → fallback to pure ensemble classification
            if pred is None:
                
                img_resized = torch.nn.functional.interpolate(
                    img.unsqueeze(0),
                    size=(224,224)
                )
                
                ensemble_probs = None
                
                for model in classifier_models.values():
                    output = model(img_resized.to(device))
                    probs = torch.softmax(output, dim=1)
                    
                    if ensemble_probs is None:
                        ensemble_probs = probs
                    else:
                        ensemble_probs += probs
                
                ensemble_probs /= len(classifier_models)
                pred = torch.argmax(ensemble_probs, dim=1).item()
            
            all_preds.append(pred)
            all_labels.append(label.item())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    return acc, precision, recall, f1


**Run Hybrid Evaluation**

In [ ]:
acc_h, precision_h, recall_h, f1_h = evaluate_hybrid(
    faster_rcnn,
    models_dict,
    val_loader,
    device
)

print("Hybrid Model Results")
print("Accuracy:", acc_h)
print("Presicion:",precision_h)
print("Recall:",recall_h)
print("F1:", f1_h)


**Final Comparison Table**

In [ ]:
final_compare = pd.DataFrame([
    {
        "Model": "Pure Ensemble",
        "Accuracy": acc_w,
        "F1 Score": f1_w
    },
    {
        "Model": "Hybrid (Detection + Ensemble)",
        "Accuracy": acc_h,
        "F1 Score": f1_h
    }
])

print(final_compare)


In [ ]:
import torch

torch.cuda.empty_cache()


**Multi-Detector Box Fusion**

In [ ]:
def get_multi_detector_crops(detectors_dict, image_tensor, device, score_thresh=0.5):
    
    all_boxes = []
    
    with torch.no_grad():
        for det in detectors_dict.values():
            det.eval()
            outputs = det([image_tensor.to(device)])
            
            boxes = outputs[0]["boxes"]
            scores = outputs[0]["scores"]
            
            for box, score in zip(boxes, scores):
                if score > score_thresh:
                    all_boxes.append(box.int())
    
    crops = []
    
    for box in all_boxes:
        x1, y1, x2, y2 = box
        crop = image_tensor[:, y1:y2, x1:x2]
        crops.append(crop)
    
    return crops


**Weighted Classification on Crops**

In [ ]:
def classify_crops_weighted(crops, models_dict, weights_dict, device):
    
    final_probs = []
    
    for crop in crops:
        
        if crop.shape[1] < 10 or crop.shape[2] < 10:
            continue
        
        crop = F.interpolate(
            crop.unsqueeze(0),
            size=(224,224)
        )
        
        ensemble_probs = None
        
        for name, model in models_dict.items():
            model.eval()
            output = model(crop.to(device))
            probs = torch.softmax(output, dim=1)
            
            weight = weights_dict[name]
            
            if ensemble_probs is None:
                ensemble_probs = weight * probs
            else:
                ensemble_probs += weight * probs
        
        final_probs.append(ensemble_probs)
    
    return final_probs


**True Hybrid Inference**

In [ ]:
def hybrid_multi_inference(image_tensor, detectors_dict, classifier_models, weights_dict, device):
    
    crops = get_multi_detector_crops(detectors_dict, image_tensor, device)
    
    particle_probs = classify_crops_weighted(
        crops,
        classifier_models,
        weights_dict,
        device
    )
    
    if len(particle_probs) == 0:
        return None
    
    total = sum(particle_probs)
    avg = total / len(particle_probs)
    
    final_class = torch.argmax(avg, dim=1)
    
    return final_class.item()


Evaluate Hybrid Multi-Detector


In [ ]:
def evaluate_hybrid_multi(detectors_dict, classifier_models, weights_dict, val_loader, device):
    
    all_preds = []
    all_labels = []
    
    for imgs, labels in val_loader:
        
        for img, label in zip(imgs, labels):
            
            pred = hybrid_multi_inference(
                img,
                detectors_dict,
                classifier_models,
                weights_dict,
                device
            )
            
            if pred is None:
                continue
            
            all_preds.append(pred)
            all_labels.append(label.item())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    return acc, precision, recall, f1, all_labels, all_preds


**detectors_dict Example**

In [ ]:
detectors_dict = {
    "FasterRCNN": faster_rcnn,
    "RetinaNet": retinanet,
    "YOLO26": yolo_model
}


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import os

acc_hy, prec_hy, rec_hy, f1_hy, y_true_hy, y_pred_hy = evaluate_hybrid_multi(
    detectors_dict,
    models_dict,
    model_weights,
    val_loader,
    device
)

print("Hybrid Multi-Detector Results")
print("Accuracy:", acc_hy)
print("Precision:",prec_hy)
print("Recall:",rec_hy)
print("F1:", f1_hy)

# Classification Report
report = classification_report(y_true_hy, y_pred_hy, target_names=class_names)
print(report)

os.makedirs("results/hybrid", exist_ok=True)

with open("results/hybrid/Hybrid_report.txt", "w") as f:
    f.write(report)

# Confusion Matrix
cm = confusion_matrix(y_true_hy, y_pred_hy, normalize="true")


plt.figure(figsize=(14,12))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    annot=True,
    fmt=".2f"
)


plt.title("Hybrid Multi-Detector Confusion Matrix")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("results/hybrid/Hybrid_confusion_matrix.png", dpi=300)
plt.show()


In [ ]:
from ultralytics import YOLO

yolo_model = YOLO("/kaggle/working/runs/detect/train/weights/best.pt")
yolo_model.fuse()


In [ ]:
detectors_dict = {
    "FasterRCNN": faster_rcnn,
    "RetinaNet": retinanet,
    "YOLO": yolo_model
}


In [ ]:
results = yolo_model.predict(
    source=image_np,
    conf=0.0001,   # extremely low
    imgsz=800,
    verbose=False
)

print("Total raw detections:", len(results[0].boxes))

if len(results[0].boxes) > 0:
    print("Max confidence:", results[0].boxes.conf.max().item())


In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/working/runs/detect/train/results.csv")
print(df.tail())


In [ ]:
yolo_model = YOLO("/kaggle/working/runs/detect/train2/weights/best.pt")
yolo_model.fuse()


**Multi-Detector Box Fusion (Final Stable Version)**

In [ ]:
import torchvision.ops as ops
import numpy as np

def get_multi_detector_crops(image_tensor, detectors_dict, device, score_thresh=0.2):
    
    all_boxes = []
    all_scores = []
    
    image_np = image_tensor.permute(1,2,0).cpu().numpy()
    
    # Ensure YOLO input is uint8
    image_np = image_np - image_np.min()
    image_np = image_np / image_np.max()
    image_np = (image_np * 255).astype("uint8")

    for name, model in detectors_dict.items():
        
        if name == "YOLO":
            
            results = model.predict(
                source=image_np,
                conf=score_thresh,
                imgsz=1024,
                verbose=False
            )
            
            if len(results[0].boxes) > 0:
                boxes = results[0].boxes.xyxy.cpu()
                scores = results[0].boxes.conf.cpu()
                all_boxes.append(boxes)
                all_scores.append(scores)

        else:
            with torch.no_grad():
                outputs = model([image_tensor.to(device)])
            
            boxes = outputs[0]["boxes"].cpu()
            scores = outputs[0]["scores"].cpu()
            
            keep = scores > score_thresh
            boxes = boxes[keep]
            scores = scores[keep]
            
            if len(boxes) > 0:
                all_boxes.append(boxes)
                all_scores.append(scores)

    if len(all_boxes) == 0:
        return []

    all_boxes = torch.cat(all_boxes, dim=0)
    all_scores = torch.cat(all_scores, dim=0)

    keep = ops.nms(all_boxes, all_scores, iou_threshold=0.5)
    fused_boxes = all_boxes[keep]

    crops = []

    for box in fused_boxes:
        x1, y1, x2, y2 = box.int()
        crop = image_tensor[:, y1:y2, x1:x2]
        if crop.shape[1] > 10 and crop.shape[2] > 10:
            crops.append(crop)

    return crops


**Weighted Ensemble Classification for Crops**

In [ ]:
import torch.nn.functional as F

def classify_crops_weighted(crops, models_dict, weights_dict, device):
    
    particle_probs = []
    
    for crop in crops:
        
        crop = F.interpolate(
            crop.unsqueeze(0),
            size=(224,224)
        )
        
        ensemble_probs = None
        
        for name, model in models_dict.items():
            
            model.eval()
            output = model(crop.to(device))
            probs = F.softmax(output, dim=1)
            
            weight = weights_dict[name]
            
            if ensemble_probs is None:
                ensemble_probs = weight * probs
            else:
                ensemble_probs += weight * probs
        
        particle_probs.append(ensemble_probs)
    
    return particle_probs


**Aggregate Particle Predictions**

In [ ]:
def aggregate_particle_predictions(particle_probs):
    
    if len(particle_probs) == 0:
        return None
    
    total = sum(particle_probs)
    avg = total / len(particle_probs)
    
    final_class = torch.argmax(avg, dim=1)
    
    return final_class.item(), avg.squeeze(0)


**Full Multi-Detector + Weighted Ensemble Hybrid**

In [ ]:
def hybrid_multi_inference(image_tensor, detectors_dict, models_dict, weights_dict, device):
    
    crops = get_multi_detector_crops(
        image_tensor,
        detectors_dict,
        device
    )
    
    particle_probs = classify_crops_weighted(
        crops,
        models_dict,
        weights_dict,
        device
    )
    
    final = aggregate_particle_predictions(particle_probs)
    
    if final is None:
        # fallback to full image weighted ensemble
        img_resized = F.interpolate(image_tensor.unsqueeze(0), size=(224,224))
        
        ensemble_probs = None
        
        for name, model in models_dict.items():
            output = model(img_resized.to(device))
            probs = F.softmax(output, dim=1)
            weight = weights_dict[name]
            
            if ensemble_probs is None:
                ensemble_probs = weight * probs
            else:
                ensemble_probs += weight * probs
        
        pred = torch.argmax(ensemble_probs, dim=1).item()
        return pred, ensemble_probs.squeeze(0)
    
    return final


**Evaluate Full Hybrid System**

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_hybrid_multi(detectors_dict, models_dict, weights_dict, loader, device):
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    for imgs, labels in loader:
        
        for img, label in zip(imgs, labels):
            
            pred, probs = hybrid_multi_inference(
                img,
                detectors_dict,
                models_dict,
                weights_dict,
                device
            )
            
            all_preds.append(pred)
            all_labels.append(label.item())
            all_probs.append(probs.cpu().numpy())
    
    acc = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average="macro")
    recall = recall_score(all_labels, all_preds, average="macro")
    f1 = f1_score(all_labels, all_preds, average="macro")
    
    return acc, precision, recall, f1, all_labels, all_preds, np.array(all_probs)


In [ ]:
acc_h, prec_h, rec_h, f1_h, y_true, y_pred, y_probs = evaluate_hybrid_multi(
    detectors_dict,
    models_dict,
    model_weights,
    val_loader,
    device
)

print("Hybrid Multi-Detector + Weighted Ensemble")
print("Accuracy:", acc_h)
print("F1:", f1_h)


**CONFUSION MATRIX**

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import os

os.makedirs("results/hybrid", exist_ok=True)

cm = confusion_matrix(y_true, y_pred, normalize="true")

plt.figure(figsize=(16,14))
sns.heatmap(
    cm,
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names,
    annot=True,
    fmt=".2f"
)

plt.title("Hybrid Multi-Detector Confusion Matrix")
plt.xticks(rotation=90)
plt.tight_layout()
plt.savefig("results/hybrid/confusion_matrix.png", dpi=300)
plt.show()

# Save numeric matrix
pd.DataFrame(cm, index=class_names, columns=class_names)\
    .to_csv("results/hybrid/confusion_matrix.csv")


**CLASSIFICATION REPORT**

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(y_true, y_pred, target_names=class_names)

print(report)

with open("results/hybrid/classification_report.txt", "w") as f:
    f.write(report)


**ROC CURVES (All 22 Classes)**

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
import numpy as np

y_true_bin = label_binarize(y_true, classes=range(len(class_names)))

plt.figure(figsize=(12,10))

for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_probs[:, i])
    roc_auc = auc(fpr, tpr)
    
    plt.plot(fpr, tpr, label=f"{class_names[i]} (AUC={roc_auc:.2f})")

plt.plot([0,1], [0,1], linestyle="--")
plt.title("Hybrid ROC Curves (22 Classes)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/hybrid/roc_curves.png", dpi=300)
plt.show()


**PRECISION–RECALL CURVES**

In [ ]:
from sklearn.metrics import precision_recall_curve

plt.figure(figsize=(12,10))

for i in range(len(class_names)):
    precision, recall, _ = precision_recall_curve(
        y_true_bin[:, i],
        y_probs[:, i]
    )
    
    plt.plot(recall, precision, label=class_names[i])

plt.title("Hybrid Precision–Recall Curves")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig("results/hybrid/pr_curves.png", dpi=300)
plt.show()


**STATISTICAL SIGNIFICANCE TEST**

In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# You must already have:
# y_true_pure, y_pred_pure from weighted ensemble evaluation

correct_pure = np.array(y_pred_pure) == np.array(y_true)
correct_hybrid = np.array(y_pred) == np.array(y_true)

# Contingency table
b = np.sum((correct_pure == True) & (correct_hybrid == False))
c = np.sum((correct_pure == False) & (correct_hybrid == True))

from statsmodels.stats.contingency_tables import mcnemar

table = [[0, b],
         [c, 0]]

result = mcnemar(table, exact=True)

print("McNemar Test p-value:", result.pvalue)


**Macro AUC & Macro PR**

In [ ]:
macro_auc = np.mean([
    auc(*roc_curve(y_true_bin[:, i], y_probs[:, i])[:2])
    for i in range(len(class_names))
])

print("Macro AUC:", macro_auc)
